# AI Speak Baseline

This notebook prepares the competition data, trains the baseline model, and exports submission files.

In [1]:
import os
import sys
import subprocess
from pathlib import Path

def find_project_root():
    markers = ['requirements.txt', 'src', 'notebooks']
    candidates = [Path.cwd(), Path.cwd().parent, Path('/content')]
    for base in list(candidates):
        if base.exists():
            candidates.extend(base.iterdir())
    seen = []
    for candidate in candidates:
        if not isinstance(candidate, Path):
            candidate = Path(candidate)
        try:
            resolved = candidate.resolve()
        except FileNotFoundError:
            continue
        if resolved in seen or not resolved.exists() or not resolved.is_dir():
            continue
        seen.append(resolved)
        if all((resolved / marker).exists() for marker in markers):
            return resolved
    raise FileNotFoundError('Project root not found. Put the whole project folder in Colab, not just the notebook.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT / 'src'))
print('PROJECT_ROOT =', PROJECT_ROOT)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')])


FileNotFoundError: Project root not found. Put the whole project folder in Colab, not just the notebook.

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'
DATA_ROOT = ARTIFACT_ROOT / 'data'
SUBMISSION_ROOT = PROJECT_ROOT / 'submission'

if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.append(str(PROJECT_ROOT / 'src'))

from ai_speak.data import FeatureConfig
from ai_speak.model import ModelConfig
from ai_speak.pipeline import TrainConfig, infer_directory, prepare_data, train_model

## Check raw archives

Make sure the zip files are in the project root before running the next cell.

In [ ]:
required = [
    'spk08_blendshapes.zip',
    'spk14_blendshapes.zip',
    'labels_aligned.zip',
]
missing = [name for name in required if not (PROJECT_ROOT / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing archives: {missing}')
print('All required archives found.')

In [ ]:
feature_config = FeatureConfig(sample_rate=16000, fps=60, n_mels=80)
data_info = prepare_data(
    raw_root=PROJECT_ROOT,
    output_root=DATA_ROOT,
    feature_config=feature_config,
    extract_avatar=False,
    force=False,
    validation_ratio=0.15,
    seed=42,
)
data_info

In [ ]:
model_config = ModelConfig(
    input_dim=data_info['feature_dim'],
    hidden_dim=256,
    num_layers=6,
    kernel_size=3,
    dropout=0.15,
    speaker_count=len(data_info['speakers']),
    vocab_size=data_info['vocab_size'],
    use_text=True,
)

train_config = TrainConfig(
    batch_size=8,
    epochs=40,
    learning_rate=3e-4,
    weight_decay=1e-3,
    validation_ratio=0.15,
    velocity_loss_weight=0.4,
    early_stopping_patience=8,
    num_workers=2,
)

train_result = train_model(
    data_root=DATA_ROOT,
    artifact_root=ARTIFACT_ROOT,
    feature_config=feature_config,
    model_config=model_config,
    train_config=train_config,
    device=None,
)
train_result

## Inference

Put test `.wav` files in `test_inputs/`.
If transcript text is available, add one `.txt` file with the same stem for each `.wav` file.

In [ ]:
TEST_INPUT_DIR = PROJECT_ROOT / 'test_inputs'
TEST_INPUT_DIR.mkdir(exist_ok=True)
SUBMISSION_ROOT.mkdir(exist_ok=True)

if not list(TEST_INPUT_DIR.glob('*.wav')):
    print('Add test wav files to', TEST_INPUT_DIR)
else:
    meta = infer_directory(
        checkpoint_path=train_result['best_checkpoint'],
        input_dir=TEST_INPUT_DIR,
        output_dir=SUBMISSION_ROOT,
        fps_out=60,
        device=None,
        default_speaker='spk08',
        smoothing_alpha=0.15,
    )
    print(json.dumps(meta, indent=2, ensure_ascii=False))

In [ ]:
import zipfile

bundle_path = PROJECT_ROOT / 'submission_bundle.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(SUBMISSION_ROOT.glob('*')):
        archive.write(path, arcname=path.name)
print('Created:', bundle_path)